# Heart Disease Risk — EDA & Modeling

**You:** Pharmacist + MS Data Science · **Pace:** ~2 hr/day

Run `python -m src.load_data` from project root before this notebook.

In [2]:
pip install matplotlib

     |████████████████████████████████| 7.8 MB 2.2 MB/s eta 0:00:01
     |████████████████████████████████| 249 kB 8.4 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 6.6 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 14.2 MB/s eta 0:00:01
     |████████████████████████████████| 122 kB 24.5 MB/s eta 0:00:01
You should consider upgrading via the '/Users/barringtonmiller/Data Science Learning/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install seaborn

     |████████████████████████████████| 294 kB 1.4 MB/s eta 0:00:01
     |████████████████████████████████| 10.8 MB 18.1 MB/s eta 0:00:01
     |████████████████████████████████| 349 kB 32.9 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 20.9 MB/s eta 0:00:01
You should consider upgrading via the '/Users/barringtonmiller/Data Science Learning/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install requests

  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
     |████████████████████████████████| 131 kB 1.9 MB/s eta 0:00:01
     |████████████████████████████████| 299 kB 9.6 MB/s eta 0:00:01
     |████████████████████████████████| 72 kB 2.8 MB/s  eta 0:00:01
     |████████████████████████████████| 135 kB 27.6 MB/s eta 0:00:01
You should consider upgrading via the '/Users/barringtonmiller/Data Science Learning/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
You should consider upgrading via the '/Users/barringtonmiller/Data Science Learning/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns

from src.config import FEATURE_LABELS, FIGURES_DIR
from src.load_data import load_processed

sns.set_theme(style="whitegrid")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

/Users/barringtonmiller/Data Science Learning/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [14]:
df = load_processed()
df.head()

ArrowKeyError: A type extension with name pandas.period already defined

In [ ]:
print(df.shape)
print(df.isna().sum())
df["target_label"].value_counts(normalize=True)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df["target_label"].value_counts().plot(kind="bar", ax=ax, color=["#4C78A8", "#E45756"])
ax.set_title("Class balance — coronary disease")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_balance.png", dpi=150)
plt.show()

## Pharmacy-relevant features

Compare **cholesterol**, **blood pressure**, and **fasting glucose proxy** by outcome.

In [ ]:
pharm_cols = ["chol", "trestbps", "fbs"]
plot_df = df.melt(
    id_vars="target_label",
    value_vars=pharm_cols,
    var_name="feature",
    value_name="value",
)
plot_df["feature"] = plot_df["feature"].map(FEATURE_LABELS)

g = sns.catplot(
    data=plot_df, x="target_label", y="value", col="feature",
    kind="box", sharey=False, height=4, aspect=1.1,
)
g.fig.suptitle("Lipids, BP, and fasting glucose proxy by disease status", y=1.02)
plt.savefig(FIGURES_DIR / "pharmacy_features_by_outcome.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
num_cols = [c for c in df.columns if c not in ("target", "target_label")]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="RdBu_r", center=0, ax=ax, square=True)
ax.set_title("Feature correlation matrix")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_heatmap.png", dpi=150)
plt.show()

## Your notes (Day 2–3)

Write 5 clinical observations here, e.g.:

1. ...
2. ...

## After training (`python -m src.train`)

Load feature importance from the saved Random Forest or inspect logistic coefficients.

In [ ]:
import joblib
from src.config import MODELS_DIR
from src.train import FEATURE_COLS

model_path = MODELS_DIR / "best_model.joblib"
import pandas as pd

if model_path.exists():
    pipe = joblib.load(model_path)
    clf = pipe.named_steps["clf"]
    if hasattr(clf, "feature_importances_"):
        imp = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
        imp.plot(kind="barh", figsize=(8, 5), title="Random Forest feature importance")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "feature_importance.png", dpi=150)
        plt.show()
    else:
        print("Best model is logistic regression — inspect coefficients in Day 6.")
else:
    print("Run: python -m src.train")